In [ ]:
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer

df_occ = pd.read_csv('Datasets/Unified_Occupation_Dataset.csv')
valid_types = ['TechnologySkills', 'Skill', 'Knowledge']
df_occ = df_occ[df_occ['Feature_Type'].isin(valid_types)]

def preserve_tech_syntax(text):
    import re
    if not isinstance(text, str): return ""
    return ' '.join(re.sub(r'[^a-z0-9\+\#\.]', ' ', text.lower()).split())

master_vocab = list(set(df_occ['Feature_Name'].apply(preserve_tech_syntax)))

aligner = SentenceTransformer('all-mpnet-base-v2')
ontology_embs = aligner.encode(master_vocab, convert_to_tensor=True)

torch.save({'vocab': master_vocab, 'embeddings': ontology_embs}, 'models/ontology_tensor.pt')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3188.93it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

vocab_path = "models_&_files/master_vocab.pkl"
with open(vocab_path, "rb") as f:
    master_vocab = pickle.load(f)
print(f"Loaded vocabulary size: {len(master_vocab)}")

dataset_file = "Datasets/Unified_Resume_Salary_Dataset.csv"
df = pd.read_csv(dataset_file)
y_full_log = df["Log_Salary"].values

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["Aligned_Text"], y_full_log, test_size=0.2, random_state=42
)

tfidf_vec = TfidfVectorizer(
    vocabulary=master_vocab, ngram_range=(1, 3),
    token_pattern=r'(?u)\.?[a-z0-9][a-z0-9\+\#\.]*',
    norm=None, binary=True
)

print("Fitting TF-IDF vectorizer...")
tfidf_vec.fit(X_train_text)  

output_path = "models_&_files/tfidf_vectorizer.pkl"
with open(output_path, "wb") as f:
    pickle.dump(tfidf_vec, f)

print(f"TF-IDF vectorizer saved successfully to {output_path}")

Loaded vocabulary size: 366
Fitting TF-IDF vectorizer...
TF-IDF vectorizer saved successfully to models/tfidf_vectorizer.pkl


In [ ]:
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from feature_pipeline import preserve_tech_syntax

model = SentenceTransformer('models_&_files/mpnet-base-onet-recsys-finetuned')

df = pd.read_csv("Datasets/Unified_Occupation_Dataset.csv")
valid_types = ['TechnologySkills', 'Skill', 'Knowledge']
df = df[df['Feature_Type'].isin(valid_types)]
df['Clean_Feature'] = df['Feature_Name'].apply(preserve_tech_syntax)

job_groups = (
    df.groupby(['ONET_SOC_Code', 'Title'])['Clean_Feature']
    .apply(lambda x: ' '.join(pd.unique(x)))
    .reset_index()
)

job_groups['Skills_Text'] = (
    job_groups['Title'] + " " + job_groups['Clean_Feature']
)

embeddings = model.encode(job_groups['Skills_Text'].tolist(), normalize_embeddings=True, show_progress_bar=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(np.array(embeddings).astype('float32'))

faiss.write_index(index, 'models_&_files/onet_faiss.index')
job_groups[['ONET_SOC_Code', 'Title']].to_pickle('models_&_files/onet_metadata.pkl')

print(f"Done. Indexed {index.ntotal} jobs.")

Batches: 100%|██████████| 28/28 [00:14<00:00,  1.98it/s]


Done. Indexed 875 jobs.
